# All Routes Discrete-Event Simulation

Core SimPy model for electric bus operations across candidate Historical Peninsula bus routes.

Input data files are expected locally when rerunning this notebook.


In [ ]:
!pip install simpy

In [ ]:
import simpy
import pandas as pd
import numpy as np
import random

# RouteSegment sınıfı
class RouteSegment:
    def __init__(self, distance_km, slope_percent):
        self.distance_km = distance_km
        self.slope_percent = slope_percent

# Yardımcı fonksiyonlar
def format_time(minutes):
    if minutes is None:
        return "N/A"
    hours = int(minutes // 60)
    mins = int(minutes % 60)
    return f"{hours:02d}:{mins:02d}"

def calculate_confidence_interval(data):
    mean = np.mean(data)
    std = np.std(data, ddof=1)
    ci = 1.96 * (std / np.sqrt(len(data)))
    return mean, std, np.var(data, ddof=1), (mean - ci, mean + ci)

# Otobüs sınıfı
class Bus:
    def __init__(self, env, name, route_segments, depot_to_start, rounds, capacity, battery_capacity, co2_per_kwh, bus_area=22):
        self.env = env
        self.name = name
        self.route_segments = route_segments
        self.depot_to_start = depot_to_start
        self.rounds = rounds
        self.capacity = capacity
        self.bus_area = bus_area

        self.passengers = 0
        self.battery_capacity = battery_capacity
        self.battery_level = battery_capacity
        self.co2_per_kwh = co2_per_kwh

        self.total_co2_emissions = 0
        self.total_energy_consumed = 0
        self.total_energy_charged = 0
        self.min_battery_level = battery_capacity
        self.charge_count = 0
        self.passenger_counts = []
        self.routes_completed = 0
        self.total_passenger_transported = 0
        self.end_of_service_time = None
        self.total_stop_time = 0
        self.action = env.process(self.run())

    def slope_correction(self, slope):
        return 1 + (slope / 10) if slope > 0 else max(0.7, 1 + (slope / 25))

    def consumption_rate(self, passenger_mass, slope):
        base = np.interp(passenger_mass, [0, 1000, 2000, 4000, 5700], [0.8, 0.9, 1.0, 1.2, 1.4])
        return max(base * self.slope_correction(slope), 0.6)

    def travel_segment(self, segment, passenger_mass):
        yield self.env.timeout((segment.distance_km / 30) * 60)

        leaving = random.randint(0, min(self.passengers, 10))
        self.passengers -= leaving
        boarding = min(random.randint(5, 20), self.capacity - self.passengers)
        self.passengers += boarding

        passenger_mass = self.passengers * 70
        self.passenger_counts.append(self.passengers)
        self.total_passenger_transported += boarding

        density = self.passengers / self.bus_area
        boarding_time = boarding * np.interp(density, [0, 1, 2, 4, 6], [0.89, 1.13, 1.37, 1.67, 2.03])
        alighting_time = leaving * np.interp(density, [0, 1, 2, 4, 6], [0.59, 0.92, 1.11, 1.89, 5.92])
        wait_time = max(boarding_time, alighting_time) + 8.8
        self.total_stop_time += wait_time / 60
        yield self.env.timeout(wait_time / 60)

        consumption = self.consumption_rate(passenger_mass, segment.slope_percent) * segment.distance_km
        self.battery_level -= consumption
        self.total_energy_consumed += consumption
        self.total_co2_emissions += consumption * self.co2_per_kwh
        self.min_battery_level = min(self.min_battery_level, self.battery_level)

    def charge(self):
        to_charge = self.battery_capacity * 0.8 - self.battery_level
        self.total_energy_charged += to_charge
        self.charge_count += 1
        yield self.env.timeout(60)
        self.battery_level = self.battery_capacity * 0.8

    def run(self):
        yield from self.travel_segment(self.depot_to_start, 0)
        for _ in range(self.rounds):
            route_distance = sum(s.distance_km for s in self.route_segments) * 2
            avg_mass = np.mean(self.passenger_counts) * 70 if self.passenger_counts else 0
            est_need = self.consumption_rate(avg_mass, 0) * route_distance + 50

            if self.battery_level < est_need:
                yield from self.travel_segment(RouteSegment(self.depot_to_start.distance_km, -self.depot_to_start.slope_percent), 0)
                yield from self.charge()
                yield from self.travel_segment(self.depot_to_start, 0)

            for seg in self.route_segments:
                yield from self.travel_segment(seg, self.passengers * 70)
            for seg in reversed(self.route_segments):
                yield from self.travel_segment(RouteSegment(seg.distance_km, -seg.slope_percent), self.passengers * 70)

            self.passengers = 0
            self.routes_completed += 1

        yield from self.travel_segment(RouteSegment(self.depot_to_start.distance_km, -self.depot_to_start.slope_percent), 0)
        self.end_of_service_time = self.env.now

# Simülasyon fonksiyonu
def run_simulation(seed, route_segments):
    random.seed(seed)
    np.random.seed(seed)
    env = simpy.Environment(initial_time=360)
    depot_to_start = RouteSegment(2.1, -0.5)

    bus1 = Bus(env, "BYD K8", route_segments, depot_to_start, 6, 90, 350, 0.233, 22)
    bus2 = Bus(env, "Karsan e-Atak", route_segments, depot_to_start, 6, 52, 220, 0.233, 17)

    env.run(until=10000)
    total_stops = len(route_segments) * 2 * 6

    return {
        "BYD K8_Passengers": bus1.total_passenger_transported,
        "BYD K8_Energy_Consumed": bus1.total_energy_consumed,
        "BYD K8_CO2": bus1.total_co2_emissions,
        "BYD K8_EndTime": format_time(bus1.end_of_service_time),
        "BYD K8_TotalServiceTime": bus1.end_of_service_time - 360 if bus1.end_of_service_time else None,
        "BYD K8_AvgStopTime": bus1.total_stop_time / total_stops if total_stops else None,
        "BYD K8_EnergyPerPassenger": bus1.total_energy_consumed / bus1.total_passenger_transported if bus1.total_passenger_transported else None,
        "BYD K8_ChargeCount": bus1.charge_count,
        "BYD K8_MinBatteryLevel": bus1.min_battery_level,

        "Karsan_Passengers": bus2.total_passenger_transported,
        "Karsan_Energy_Consumed": bus2.total_energy_consumed,
        "Karsan_CO2": bus2.total_co2_emissions,
        "Karsan_EndTime": format_time(bus2.end_of_service_time),
        "Karsan_TotalServiceTime": bus2.end_of_service_time - 360 if bus2.end_of_service_time else None,
        "Karsan_AvgStopTime": bus2.total_stop_time / total_stops if total_stops else None,
        "Karsan_EnergyPerPassenger": bus2.total_energy_consumed / bus2.total_passenger_transported if bus2.total_passenger_transported else None,
        "Karsan_ChargeCount": bus2.charge_count,
        "Karsan_MinBatteryLevel": bus2.min_battery_level,
    }

# Ana çalıştırma
df_all = pd.read_excel("28tve32_temizlenmis.xlsx")  # <- buraya dosya adını doğru girin!!!
summary_records = []

for line_code in df_all["Hat Kodu"].unique():
    df_line = df_all[df_all["Hat Kodu"] == line_code].reset_index(drop=True)
    segments = [RouteSegment(row["Route Distance (km)"], row["Slope (%)"]) for _, row in df_line.iterrows()]
    results = [run_simulation(seed, segments) for seed in range(1, 101)]
    df_results = pd.DataFrame(results)

    for col in df_results.columns:
        if "EndTime" in col:
            continue
        clean_col = df_results[col].dropna()
        if len(clean_col) == 0:
            continue
        mean, std, var, ci = calculate_confidence_interval(clean_col)
        summary_records.append({
            "Hat Kodu": line_code,
            "Metrik": col,
            "Mean": mean,
            "Std Dev": std,
            "Variance": var,
            "95% CI Lower": ci[0],
            "95% CI Upper": ci[1]
        })

summary_table = pd.DataFrame(summary_records)
summary_table.to_excel("tum_hatlar_simulasyon_ozeti.xlsx", index=False)
print("summary 'tum_hatlar_simulasyon_ozeti.xlsx' recorded.")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Excel dosyasını yükle
df_summary = pd.read_excel("tum_hatlar_simulasyon_ozeti.xlsx")

# Minimum batarya seviyeleri metriklerini filtrele
battery_metrics = ["BYD K8_MinBatteryLevel", "Karsan_MinBatteryLevel"]
df_battery = df_summary[df_summary["Metrik"].isin(battery_metrics)]

# Pivot tablo oluştur
df_battery_pivot = df_battery.pivot(index="Hat Kodu", columns="Metrik", values="Mean").reset_index()

# Stil ayarları
sns.set(style="whitegrid")

# BYD grafiği
plt.figure(figsize=(14, 5))
sorted_byd = df_battery_pivot.sort_values("BYD K8_MinBatteryLevel")
sns.barplot(x="Hat Kodu", y="BYD K8_MinBatteryLevel", data=sorted_byd, palette="viridis")
plt.title("🔋 BYD K8 - Minimum Batarya Seviyesi (Hat Bazlı)")
plt.xlabel("Hat Kodu")
plt.ylabel("Minimum Batarya Seviyesi (kWh)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("BYD_K8_MinBatteryLevel.png")
plt.show()

# Karsan grafiği
plt.figure(figsize=(14, 5))
sorted_karsan = df_battery_pivot.sort_values("Karsan_MinBatteryLevel")
sns.barplot(x="Hat Kodu", y="Karsan_MinBatteryLevel", data=sorted_karsan, palette="crest")
plt.title("🔋 Karsan e-Atak - Minimum Batarya Seviyesi (Hat Bazlı)")
plt.xlabel("Hat Kodu")
plt.ylabel("Minimum Batarya Seviyesi (kWh)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("Karsan_MinBatteryLevel.png")
plt.show()


In [ ]:
class Simulation:
    def __init__(self, seed, route_segments, depot_distance_km=2.1, depot_slope=-0.5):
        self.seed = seed
        self.route_segments = route_segments
        self.env = simpy.Environment(initial_time=360)
        random.seed(seed)
        np.random.seed(seed)

        self.depot_to_start = RouteSegment(
            from_stop=-1,
            to_stop=0,
            distance_km=depot_distance_km,
            slope_percent=depot_slope,
            from_name="Edirnekapı Garajı",
            to_name="Topkapı"
        )

        self.bus1 = Bus(self.env, "BYD K8", self.route_segments, self.depot_to_start, 6, 90, 350, 0.233, 22)
        self.bus2 = Bus(self.env, "Karsan e-Atak", self.route_segments, self.depot_to_start, 6, 52, 220, 0.233, 17)

    def run(self, until=10000):
        self.env.run(until=until)
        total_stops = len(self.route_segments) * 2 * 6  # gidiş + dönüş * 6 tur

        return {
            "BYD K8_Passengers": self.bus1.total_passenger_transported,
            "BYD K8_Energy_Consumed": self.bus1.total_energy_consumed,
            "BYD K8_CO2": self.bus1.total_co2_emissions,
            "BYD K8_EndTime": format_time(self.bus1.end_of_service_time),
            "BYD K8_TotalServiceTime": self.bus1.end_of_service_time - 360 if self.bus1.end_of_service_time else None,
            "BYD K8_AvgStopTime": self.bus1.total_stop_time / total_stops if total_stops else None,
            "BYD K8_EnergyPerPassenger": self.bus1.total_energy_consumed / self.bus1.total_passenger_transported if self.bus1.total_passenger_transported else None,
            "BYD K8_ChargeCount": self.bus1.charge_count,
            "BYD K8_MinBatteryLevel": self.bus1.min_battery_level,

            "Karsan_Passengers": self.bus2.total_passenger_transported,
            "Karsan_Energy_Consumed": self.bus2.total_energy_consumed,
            "Karsan_CO2": self.bus2.total_co2_emissions,
            "Karsan_EndTime": format_time(self.bus2.end_of_service_time),
            "Karsan_TotalServiceTime": self.bus2.end_of_service_time - 360 if self.bus2.end_of_service_time else None,
            "Karsan_AvgStopTime": self.bus2.total_stop_time / total_stops if total_stops else None,
            "Karsan_EnergyPerPassenger": self.bus2.total_energy_consumed / self.bus2.total_passenger_transported if self.bus2.total_passenger_transported else None,
            "Karsan_ChargeCount": self.bus2.charge_count,
            "Karsan_MinBatteryLevel": self.bus2.min_battery_level,
        }

In [ ]:


# Excel dosyasını yükle
df = pd.read_excel("tum_hatlar_simulasyon_ozeti.xlsx")

# Seçilen metrikler
selected_metrics = [
    "BYD K8_Passengers", "BYD K8_Energy_Consumed", "BYD K8_EnergyPerPassenger", "BYD K8_CO2",
    "Karsan_Passengers", "Karsan_Energy_Consumed", "Karsan_EnergyPerPassenger", "Karsan_CO2",
    "BYD K8_ChargeCount", "Karsan_ChargeCount", "BYD K8_MinBatteryLevel", "Karsan_MinBatteryLevel"
]

# Veriyi filtrele ve pivotla
df_filtered = df[df["Metrik"].isin(selected_metrics)]
df_pivot = df_filtered.pivot(index="Hat Kodu", columns="Metrik", values="Mean").reset_index()

# CO2 per passenger metriklerini hesapla
df_pivot["BYD K8_CO2PerPassenger"] = df_pivot["BYD K8_CO2"] / df_pivot["BYD K8_Passengers"]
df_pivot["Karsan_CO2PerPassenger"] = df_pivot["Karsan_CO2"] / df_pivot["Karsan_Passengers"]

# Grafik üretimi
all_metrics_to_plot = selected_metrics + ["BYD K8_CO2PerPassenger", "Karsan_CO2PerPassenger"]
sns.set(style="whitegrid")

for metric in all_metrics_to_plot:
    if metric not in df_pivot.columns:
        print(f"⚠️ {metric} verisi eksik.")
        continue
    plt.figure(figsize=(14, 5))
    sorted_df = df_pivot.sort_values(by=metric, ascending=(metric not in ["Passengers", "CO2"]))
    sns.barplot(x="Hat Kodu", y=metric, data=sorted_df, palette="viridis")
    plt.xticks(rotation=45)
    plt.title(f"{metric} - Hat Bazlı Karşılaştırma")
    plt.xlabel("Hat Kodu")
    plt.ylabel(metric)
    plt.tight_layout()
    plt.savefig(f"{metric.replace(' ', '_')}.png")
    plt.show()
